In [1]:
# Code to extract the annual data from 1987 to 2022 for the air quality per US state.

# libraries
import numpy as np
import pandas as pd
import datetime as dt
import re
import matplotlib.pyplot as plt
from os.path import abspath, exists

In [2]:
# function to read file into a dataframe
def readFile(filename):

    # read the data file
    fileName = "data/airQuality/" + filename
    fPath = abspath(fileName)
    #print(f"fPath{fPath}")
    assert exists(fPath), f"{fileName} doesn't exist"
    df = pd.read_csv(fPath)
    return df

In [3]:
fileNamePreFix = "annual_aqi_by_county_"
yearStart = 1987
yearEnd = 2022

dfList = []
for y in range(yearStart, yearEnd+1):
    fileName = fileNamePreFix + str(y) + ".csv"
    df = readFile(fileName)
    df = df[~df['State'].str.lower().str.contains('country of mexico')]
    df = df[~df['State'].str.lower().str.contains('puerto')]
    df = df[~df['State'].str.lower().str.contains('islands')]
    df = df[~df['State'].str.lower().str.contains('guam')]
    df = df[~df['State'].str.lower().str.contains('canada')]
    df = df[~df['State'].str.lower().str.contains('south dakota')]
    df = df[~df['State'].str.lower().str.contains('vermont')]
    df = df[~df['State'].str.lower().str.contains('^kansas', regex=True)]
    dfList.append(df)
yearDf = pd.concat(dfList)

In [4]:
display(yearDf.head())

,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10
0,Alabama,Cullman,1987,125,73,31,16,5,0,0,177,115,47,0,0,125,0,0
1,Alabama,Etowah,1987,59,40,19,0,0,0,0,70,57,39,0,0,0,0,59
2,Alabama,Jefferson,1987,365,167,158,36,4,0,0,182,105,52,94,0,158,0,113
3,Alabama,Madison,1987,237,149,60,22,6,0,0,172,108,45,0,0,237,0,0
4,Alabama,Mobile,1987,287,199,70,16,2,0,0,154,84,43,0,0,247,0,40


In [5]:
 aggregations = {
        'Days with AQI' : 'mean',
        'Good Days' : 'mean',
        'Moderate Days' : 'mean',
        'Unhealthy for Sensitive Groups Days' : 'mean',
        'Unhealthy Days' : 'mean',
        'Very Unhealthy Days' : 'mean',
        'Hazardous Days' : 'mean',
        'Max AQI' : 'mean',
        '90th Percentile AQI' : 'mean',
        'Median AQI' : 'mean',
        'Days CO' : 'mean',
        'Days NO2' : 'mean',
        'Days Ozone' : 'mean',
        'Days PM2.5' : 'mean',
        'Days PM10' : 'mean'
}

#print(f"cols = {yearDf.columns}")
    
finalDf = yearDf.groupby(['Year', 'State']).agg(aggregations).reset_index()

In [6]:
# change the string of District of Columbia
finalDf['State'] = np.where(finalDf['State'] == 'District Of Columbia', 'District of Columbia', finalDf['State'])

# add the state mapping to the final dataframe
stDf = readFile("state_abbrev.csv")
stDict = dict(zip(stDf['State'].to_list(), stDf['Abbreviation'].to_list()))
#print(f"stDict {stDict}")
finalDf['StateAbbrv'] = finalDf['State'].map(stDict)
#print(f"unique list = {finalDf['StateAbbrv'].unique()}")
ind = np.where(finalDf['StateAbbrv'].isna())[0]
#print(f"ind: {ind}")


In [7]:
# convert the year from an in to a datetime
finalDf['Year'] = pd.to_datetime(finalDf['Year'], format="%Y").dt.date
#print(f"year: {finalDf.head()}")

# need to sort by year, then state to ensure sorted out correctly
finalDf.sort_values(['Year', 'State'], inplace=True)
finalDf.reset_index(drop=True, inplace=True)
display(finalDf.head())

,Year,State,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,StateAbbrv
0,1987-01-01,Alabama,230.571429,130.714286,66.428571,24.428571,7.857143,1.142857,0.000000,170.285714,107.571429,46.285714,13.428571,0.000000,181.142857,0.0,36.000000,AL
1,1987-01-01,Alaska,281.750000,222.000000,53.000000,6.000000,0.750000,0.000000,0.000000,111.250000,59.250000,30.000000,85.750000,0.000000,37.000000,0.0,159.000000,AK
2,1987-01-01,Arizona,162.100000,88.700000,60.500000,11.900000,1.000000,0.000000,0.000000,127.100000,76.400000,42.600000,40.800000,6.100000,79.200000,0.0,36.000000,AZ
3,1987-01-01,Arkansas,184.000000,136.333333,35.333333,8.666667,3.666667,0.000000,0.000000,101.333333,64.000000,31.666667,0.000000,16.666667,101.666667,0.0,65.666667,AR
4,1987-01-01,California,302.163265,168.612245,74.918367,29.591837,17.551020,11.469388,0.020408,186.142857,110.979592,53.102041,30.571429,63.469388,190.734694,0.0,17.387755,CA


In [8]:
# convert the column names to lower case and put underscores instead of spaces for the column names where 
# spaces are found
finalDf.columns = [re.sub('[\s]', '_', i.lower()) for i in finalDf.columns]
finalDf['state'] = finalDf['state'].str.upper()


In [9]:
outFile = "data/airQuality/yearlyAirQuality.csv"
fPath = abspath(outFile)
#finalDf.to_csv(fPath, index=False)

In [10]:
# need to impute kansas, vermont and south dakota
update_df = finalDf.copy()
impute_state_list = ['KANSAS', 'VERMONT', 'SOUTH DAKOTA']

abbrv_dict = {'KANSAS':'KS', 'VERMONT' : 'VT', 'SOUTH DAKOTA':'SD'}

for s in impute_state_list:
    df = update_df[update_df['state'] == s].copy()
    df = df.iloc[-5:,:]
    df_columns = df.columns.difference(['year', 'state', 'stateabbrv'])

    df = df[df_columns].copy()

    df_mean_list = df.mean().tolist()
    df_dict = dict(zip(df_columns, df_mean_list))

    df_mean = pd.DataFrame(list(df_dict.items()))
    df_mean = df_mean.transpose()[-1:]
    df_mean.columns = df_columns
    df_mean['stateabbrv'] = abbrv_dict[s]
    df_mean.reset_index(drop=True, inplace=True)
    
    year_state_dict = {}
    year_state_dict['year'] = pd.to_datetime('20220101').date()
    year_state_dict['state'] = s
    
    df_pre_mean = pd.DataFrame(list(year_state_dict.items()))
    df_pre_mean = df_pre_mean.transpose()[-1:]
    df_pre_mean.columns = ['year', 'state']
    df_pre_mean.reset_index(drop=True, inplace=True)

    df_mean['stateabbrv'] = abbrv_dict[s]
    df_mean = pd.concat([df_pre_mean, df_mean], axis=1)
    update_df = pd.concat([update_df, df_mean])

update_df.sort_values(['year', 'state'], inplace=True)
update_df.reset_index(drop=True, inplace=True)

In [11]:
update_df.to_csv(fPath, index=False)